# Fase 6 — Visualização e Consolidação de Resultados

**Projeto**: Mineração de Padrões Frequentes em Acidentes de Rodovias Federais  
**Grupo 18**: Erik Roberto Reis Neves, Gabriel Campos Prudente, Felipe Silva Fraga Damasceno  
**Disciplina**: Mineração de Dados — UFMG  

---

## Objetivos desta fase
Esta fase é o ápice do projeto, onde as descobertas matemáticas da mineração de dados são traduzidas para inteligência de negócio acionável (*Storytelling de Dados*). Iremos produzir os artefatos visuais finais de alta resolução que responderão aos questionamentos do domínio através de:

1. Gráficos de barra, dispersão e mapas de calor para análise exploratória direcionada.
2. Histogramas e diagramas de rede espacial (*Networkx Graph*) das regras geradas pelo modelo.
3. Dashboards e acompanhamentos de evolução sazonal.
4. Exportação das melhores descobertas para incorporação em relatórios estáticos (`.csv`, `.tex`, `.md`).

**Alinhamento CRISP-DM**: *Evaluation* & *Deployment*


## Setup — Importações e Configurações

In [1]:
"""
Fase 6 -- Visualizacao e Consolidacao de Resultados (script executavel)
Storytelling de dados e geracao das 10 visualizacoes finais de alta qualidade.
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import networkx as nx
import folium
from folium.plugins import HeatMap
import warnings
import pickle
warnings.filterwarnings('ignore')

## Configuracoes

In [2]:
# Estilos bonitos e legiveis para publicacao
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['figure.dpi'] = 300
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../data/processed/')
OUTPUT_DIR = Path('../outputs/')
FIGURAS_DIR = OUTPUT_DIR / 'figuras'
TABELAS_DIR = OUTPUT_DIR / 'tabelas'
DADOS_DIR = OUTPUT_DIR / 'dados'
for d in [FIGURAS_DIR, TABELAS_DIR, DADOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Cores para gravidade
CORES_GRAVIDADE = {
    'Com Vítimas Fatais': '#e74c3c',       # Vermelho
    'Com Vítimas Feridas': '#f39c12',      # Laranja
    'Sem Vítimas': '#2ecc71',              # Verde
    'Fatal': '#e74c3c',
    'Não Fatal': '#3498db'
}

print('[OK] Imports e configuracoes realizados.')

[OK] Imports e configuracoes realizados.


## 6.0 Carregamento dos Dados para Storytelling

In [3]:
print('\n' + '='*60)
print('6.0 CARREGAMENTO DOS DADOS')
print('='*60)

# Dados brutos limpos (com variaveis originais como Lat/Lon)
try:
    df = pd.read_pickle(PROCESSED_DIR / 'df_com_clusters.pkl')
    print(f'[OK] DataFrame principal com clusters carregado (n={len(df)})')
except FileNotFoundError:
    df = pd.read_pickle(PROCESSED_DIR / 'df_limpo.pkl')
    print(f'[OK] DataFrame limpo carregado (n={len(df)})')

# Regras geradas
regras = pd.read_csv(DADOS_DIR / 'regras_completas.csv')
print(f'[OK] Regras de associacao carregadas (n={len(regras)})')

itemsets = pd.read_csv(DADOS_DIR / 'itemsets_frequentes.csv')
print(f'[OK] Itemsets frequentes carregados (n={len(itemsets)})')


6.0 CARREGAMENTO DOS DADOS
[OK] DataFrame principal com clusters carregado (n=2985)
[OK] Regras de associacao carregadas (n=7923)
[OK] Itemsets frequentes carregados (n=2467)


## 6.1 Análise Descritiva Avançada (EDA Focada Em Resultados)

In [4]:
print('\n' + '='*60)
print('6.1 ANALISE DESCRITIVA AVANCADA')
print('='*60)

# --- VISUALIZACAO 1: Distribuicao de Gravidade ---
fig, ax = plt.subplots(figsize=(10, 6))
contagem_grav = df['classificacao_acidente'].value_counts()
barras = ax.bar(contagem_grav.index, contagem_grav.values, color=[CORES_GRAVIDADE.get(x, '#95a5a6') for x in contagem_grav.index])
ax.set_title('Proporção da Gravidade dos Acidentes Registrados (MG)')
ax.set_ylabel('Número de Acidentes')
# Adicionar rotulos nas barras
for p in barras:
    height = p.get_height()
    pct = height / len(df) * 100
    ax.annotate(f'{height:,}\n({pct:.1f}%)',
                xy=(p.get_x() + p.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, max(contagem_grav.values) * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '01_distribuicao_gravidade.png', dpi=300)
plt.close()
print('[OK] Visualizacao 1: distribuicao_gravidade.png')

# --- VISUALIZACAO 2: Heatmap Temporal (Gravidade x Dia x Fase) ---
# Focar em acidentes com feridos ou fatais
df_graves = df[df['classificacao_acidente'] != 'Sem Vítimas'].copy()
dias_ordem = ['domingo', 'sábado', 'sexta-feira', 'quinta-feira', 'quarta-feira', 'terça-feira', 'segunda-feira']
fases_ordem = ['Amanhecer', 'Pleno dia', 'Anoitecer', 'Plena Noite']

tabela_cruzada = pd.crosstab(
    df_graves['dia_semana'].str.lower(), 
    df_graves['fase_dia']
)
# Reordenar se os indices existirem
dias_existentes = [d for d in dias_ordem if d in tabela_cruzada.index]
fases_existentes = [f for f in fases_ordem if f in tabela_cruzada.columns]
tabela_cruzada = tabela_cruzada.loc[dias_existentes, fases_existentes]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(tabela_cruzada, annot=True, fmt='d', cmap='YlOrRd', linewidths=1, ax=ax)
ax.set_title('Densidade de Acidentes com Vítimas\n(Dia da Semana x Fase do Dia)')
ax.set_xlabel('Fase do Dia')
ax.set_ylabel('Dia da Semana')
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '02_heatmap_temporal_gravidade.png', dpi=300)
plt.close()
print('[OK] Visualizacao 2: heatmap_temporal_gravidade.png')

# --- VISUALIZACAO 3: Top Causas Empilhadas por Gravidade ---
top_10_causas = df['causa_acidente'].value_counts().head(10).index
df_top_causas = df[df['causa_acidente'].isin(top_10_causas)]

tabela_causas = pd.crosstab(df_top_causas['causa_acidente'], df_top_causas['classificacao_acidente'])
# Ordenar pelo total
tabela_causas['Total'] = tabela_causas.sum(axis=1)
tabela_causas = tabela_causas.sort_values('Total', ascending=True).drop('Total', axis=1)

# Plotar stacked horizontal
cores_stack = [CORES_GRAVIDADE.get(col, '#95a5a6') for col in tabela_causas.columns]
fig, ax = plt.subplots(figsize=(14, 8))
tabela_causas.plot(kind='barh', stacked=True, color=cores_stack, ax=ax, width=0.8)
ax.set_title('Top 10 Causas de Acidentes por Gravidade')
ax.set_xlabel('Número de Acidentes')
ax.set_ylabel('')
# Limpar labels muito compridos
labels = [l.get_text().replace(' ou ineficiente do condutor', '...') if len(l.get_text())>40 else l.get_text() for l in ax.get_yticklabels()]
ax.set_yticklabels(labels)
ax.legend(title='Classificação', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '03_top_causas_gravidade.png', dpi=300)
plt.close()
print('[OK] Visualizacao 3: top_causas_gravidade.png')

# --- VISUALIZACAO 4: Mapa de Calor Geografico (Folium) ---
# Filtrar pontos validos
df_mapa = df.dropna(subset=['latitude', 'longitude']).copy()

if len(df_mapa) > 0:
    # Centro aproximado de MG
    mapa = folium.Map(location=[-18.512, -44.555], zoom_start=6, tiles='CartoDB positron')
    
    # Criar heatmap de acidentes fatais (peso maior) vs outros
    pontos_calor = []
    for _, row in df_mapa.iterrows():
        lat, lon = row['latitude'], row['longitude']
        peso = 3 if row.get('gravidade_binaria') == 'Fatal' else 1
        pontos_calor.append([lat, lon, peso])
    
    HeatMap(pontos_calor, radius=10, blur=15, max_zoom=1).add_to(mapa)
    
    mapa_path = FIGURAS_DIR / '04_mapa_calor_acidentes.html'
    mapa.save(mapa_path)
    print(f'[OK] Visualizacao 4: Mapa iterativo salvo em {mapa_path}')
else:
    print('[!] Sem dados validos de latitude/longitude para gerar o mapa.')


6.1 ANALISE DESCRITIVA AVANCADA


[OK] Visualizacao 1: distribuicao_gravidade.png


[OK] Visualizacao 2: heatmap_temporal_gravidade.png


[OK] Visualizacao 3: top_causas_gravidade.png


[OK] Visualizacao 4: Mapa iterativo salvo em ..\outputs\figuras\04_mapa_calor_acidentes.html


## 6.2 Visualizações do Modelo e Regras

In [5]:
print('\n' + '='*60)
print('6.2 VISUALIZACOES DO MODELO E REGRAS')
print('='*60)

# --- VISUALIZACAO 5: Histograma de Suporte dos Itemsets ---
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=itemsets, x='support', bins=50, kde=True, color='#8e44ad', ax=ax)
ax.set_title(f'Distribuição de Suporte dos Padrões (N={len(itemsets)})')
ax.set_xlabel('Suporte do Padrão')
ax.set_ylabel('Frequência (Nº de Itemsets)')
ax.axvline(x=0.05, color='red', linestyle='--', label='Limiar Min_Support (5%)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '05_distribuicao_suporte_itemsets.png', dpi=300)
plt.close()
print('[OK] Visualizacao 5: distribuicao_suporte_itemsets.png')

# --- VISUALIZACAO 6: Top 15 Regras de Ouro (Barplot) ---
top15 = regras.nlargest(15, 'lift').copy()
# Simplificar labels para o grafico
def simplificar_regra(ante, cons):
    a = ' & '.join([x.split('_', 1)[-1] if '_' in x else x for x in str(ante).split(' & ')])
    c = ' & '.join([x.split('_', 1)[-1] if '_' in x else x for x in str(cons).split(' & ')])
    if len(a) > 35: a = a[:32] + '...'
    if len(c) > 35: c = c[:32] + '...'
    return f"{a} => {c}"

top15['label_curto'] = top15.apply(lambda r: simplificar_regra(r['antecedents_str'], r['consequents_str']), axis=1)

fig, ax = plt.subplots(figsize=(14, 8))
cores_lift = sns.color_palette("viridis", len(top15))
barras = ax.barh(range(len(top15)), top15['lift'], color=cores_lift)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['label_curto'], fontsize=11)
ax.set_xlabel('Score de Associação (Lift)')
ax.set_title('Top 15 Regras Mais Fortes Extraídas Pelo Modelo')
ax.invert_yaxis()

for p in barras:
    width = p.get_width()
    ax.annotate(f' Lift: {width:.1f}',
                xy=(width, p.get_y() + p.get_height() / 2),
                xytext=(5, 0), textcoords="offset points",
                ha='left', va='center', fontweight='bold', fontsize=10)

sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '06_top15_regras_lift_limpo.png', dpi=300)
plt.close()
print('[OK] Visualizacao 6: top15_regras_lift_limpo.png')

# --- VISUALIZACAO 7: Scatter Suporte x Confianca (Aesthetics Update) ---
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    regras['support'], regras['confidence'],
    c=regras['lift'], cmap='plasma',
    s=(regras['leverage'] * 10000).clip(lower=20, upper=300), 
    alpha=0.7, edgecolors='white', linewidth=0.5
)
ax.set_title(f'Universo de Regras: Suporte × Confiança × Lift (N={len(regras)})')
ax.set_xlabel('Suporte (Frequência da Regra)')
ax.set_ylabel('Confiança (Probabilidade Condicional)')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Força da Associação (Lift)', rotation=270, labelpad=20)
# Destacar a regiao ideal
ax.axhline(y=0.8, color='green', linestyle=':', alpha=0.5)
ax.axvline(x=0.1, color='green', linestyle=':', alpha=0.5)
ax.text(0.105, 0.95, 'Regras "Sweet Spot"\n(Alta Conf. e Sup.)', color='green', fontsize=12)

sns.despine()
plt.tight_layout()
plt.savefig(FIGURAS_DIR / '07_scatter_regras_bolhas.png', dpi=300)
plt.close()
print('[OK] Visualizacao 7: scatter_regras_bolhas.png')

# --- VISUALIZACAO 8: Network Graph das Regras Fortes ---
# Pegar o top 30 para o grafo nao virar uma macarronada
top_net = regras.nlargest(30, 'lift')

G = nx.DiGraph()

for _, row in top_net.iterrows():
    antes = [x.strip() for x in row['antecedents_str'].split('&')]
    conss = [x.strip() for x in row['consequents_str'].split('&')]
    
    # Adicionar nos do antecedente conectados a um no "regra" invisivel
    # Simplificar nomes
    antes_limpos = [x.split('_', 1)[-1] if '_' in x and not x.startswith('fim_de_semana') else x for x in antes]
    conss_limpos = [x.split('_', 1)[-1] if '_' in x and not x.startswith('fim_de_semana') else x for x in conss]
    
    for a in antes_limpos:
        for c in conss_limpos:
            peso = row['lift'] / 5.0 # normalizar a espessura
            if G.has_edge(a, c):
                G[a][c]['weight'] += peso
            else:
                G.add_edge(a, c, weight=peso)

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.5, seed=42)

# Centralidade para tamanho dos nos
d = dict(G.degree)
node_sizes = [v * 300 for v in d.values()]

# Separar cores para causas vs tipos vs outros
node_colors = []
for node in G.nodes():
    if 'Colisão' in node or 'Saída' in node or 'Tombamento' in node:
        node_colors.append('#e74c3c') # Acidentes
    elif 'Fatais' in node or 'Feridas' in node:
        node_colors.append('#8e44ad') # Gravidade
    else:
        node_colors.append('#3498db') # Condicoes/Causas

edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, width=weights, edge_color='#bdc3c7', arrows=True, arrowsize=20, arrowstyle='->', ax=ax)

# Labels
label_pos = pos
nx.draw_networkx_labels(G, label_pos, font_size=9, font_weight='bold', font_family='sans-serif', ax=ax)

ax.set_title('Rede Complexa de Associações (Top 30 Regras por Lift)')
ax.axis('off')

# Legenda customizada
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color='#e74c3c', label='Tipo de Acidente')
purple_patch = mpatches.Patch(color='#8e44ad', label='Gravidade')
blue_patch = mpatches.Patch(color='#3498db', label='Condição/Causa')
plt.legend(handles=[red_patch, purple_patch, blue_patch], loc='lower right')

plt.tight_layout()
plt.savefig(FIGURAS_DIR / '08_rede_regras.png', dpi=300)
plt.close()
print('[OK] Visualizacao 8: rede_regras.png')

# --- VISUALIZACAO 9: Evolucao Temporal ---
try:
    with open(PROCESSED_DIR / 'rules_por_periodo.pkl', 'rb') as f:
        rules_por_periodo = pickle.load(f)
    
    # Pegar as duas regras mais estaveis e plotar Lift delas
    # Eu sei que regras estaveis estao em regras_estaveis.csv se a Fase 5 rodou perfeitamente
    df_estaveis = pd.read_csv(TABELAS_DIR / 'regras_estaveis.csv')
    top2_estaveis = df_estaveis.head(2)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    meses = sorted(list(rules_por_periodo.keys()))
    
    cores = ['#2980b9', '#c0392b']
    
    for idx, (_, regra_alvo) in enumerate(top2_estaveis.iterrows()):
        a_str = regra_alvo['antecedents_str']
        c_str = regra_alvo['consequents_str']
        
        y_lift = []
        for mes in meses:
            rp = rules_por_periodo[mes]
            # Formatar keys (os dados la dentro podem ter diferencas minimas nos types, entao comparamos substrings exatas ou set equals)
            # Para simplificar, vou extrair a confianca da regra no mes
            mask = (rp['antecedents'].astype(str).str.contains(a_str.split('&')[0].strip())) & \
                   (rp['consequents'].astype(str).str.contains(c_str.split('&')[0].strip()))
            
            if len(rp[mask]) > 0:
                y_lift.append(rp[mask].iloc[0]['lift'])
            else:
                y_lift.append(np.nan)
                
        label_curta = f"{a_str.split('_')[-1][:15]} -> {c_str.split('_')[-1][:15]}"
        ax.plot(meses, y_lift, marker='o', linewidth=3, markersize=8, color=cores[idx], label=label_curta)
    
    ax.set_title('Evolução do Lift de Padrões Sistêmicos ao Longo dos Meses')
    ax.set_xlabel('Mês (Período de Extração)')
    ax.set_ylabel('Lift (Força da Associação)')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    sns.despine()
    plt.tight_layout()
    plt.savefig(FIGURAS_DIR / '09_evolucao_temporal_regras.png', dpi=300)
    plt.close()
    print('[OK] Visualizacao 9: evolucao_temporal_regras.png')
except Exception as e:
    print(f'[!] Nao foi possivel gerar a visualizacao temporal. Erro: {e}')

# --- VISUALIZACAO 10: Perfis de Cluster (Radar Chart) ---
if 'cluster' in df.columns:
    print('\nCalculando perfis para Radar Chart...')
    # Criar metricas continuas de interesse para plotar no radar
    # Transformar algumas categoricas em dummy/numerico pra tirar a media
    df_radar = df[['cluster']].copy()
    df_radar['Fatal'] = (df['gravidade_binaria'] == 'Fatal').astype(int)
    df_radar['FimDeSemana'] = (df['fim_de_semana'] == 'Sim').astype(int)
    df_radar['Noite'] = (df['fase_dia'] == 'Plena Noite').astype(int)
    df_radar['Chuva'] = (df['condicao_metereologica'] == 'Chuva').astype(int)
    
    if 'faixa_km' in df.columns:
        df_radar['KM_Alto'] = (df['faixa_km'] == 'KM_400+').astype(int)
    
    radar_agregado = df_radar.groupby('cluster').mean()
    
    # O radar chart classico em matplotlib
    labels = radar_agregado.columns.tolist()
    num_vars = len(labels)
    
    # Distribuir angulos
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1] # Fechar o poligono
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    # Cores
    colors = ['#1abc9c', '#e67e22', '#9b59b6', '#34495e']
    
    for i, row in radar_agregado.iterrows():
        valores = row.values.tolist()
        valores += valores[:1] # Fechar o poligono
        
        ax.plot(angles, valores, color=colors[i], linewidth=2, linestyle='solid', label=f'Cluster {i}')
        ax.fill(angles, valores, color=colors[i], alpha=0.25)
        
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=12)
    ax.set_title('Perfil Comparativo dos Clusters de Acidentes', y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    
    plt.tight_layout()
    plt.savefig(FIGURAS_DIR / '10_perfil_clusters_radar.png', dpi=300)
    plt.close()
    print('[OK] Visualizacao 10: perfil_clusters_radar.png')


6.2 VISUALIZACOES DO MODELO E REGRAS


[OK] Visualizacao 5: distribuicao_suporte_itemsets.png


[OK] Visualizacao 6: top15_regras_lift_limpo.png


[OK] Visualizacao 7: scatter_regras_bolhas.png


[OK] Visualizacao 8: rede_regras.png


[OK] Visualizacao 9: evolucao_temporal_regras.png

Calculando perfis para Radar Chart...


[OK] Visualizacao 10: perfil_clusters_radar.png


## 6.3 Exportacoes Para Latex / Relatorio Final

In [6]:
print('\n' + '='*60)
print('6.3 EXPORTACOES PARA RELATORIO ACADEMICO')
print('='*60)

regras_top_export = regras.nlargest(15, 'lift')

# Exportar como LaTex
try:
    # Selecionar colunas chaves
    cols = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']
    regras_top_export[cols].to_latex(TABELAS_DIR / 'regras_top15_latex.tex', index=False, float_format="%.3f")
    
    # Exportar como Markdown pra colar facil
    regras_top_export[cols].to_markdown(TABELAS_DIR / 'regras_top15_markdown.md', index=False)
    print(f'[OK] Tabelas LaTeX e Markdown exportadas para {TABELAS_DIR}')
except Exception as e:
    print(f'[!] Erro ao exportar para LaTeX: {e}')

print('\n' + '='*60)
print('[OK] FASE 6 CONCLUIDA COM SUCESSO')
print('='*60)


6.3 EXPORTACOES PARA RELATORIO ACADEMICO


[!] Erro ao exportar para LaTeX: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

[OK] FASE 6 CONCLUIDA COM SUCESSO


## 6.V2 Visualizações (Pacote A)

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, str(Path('..') / 'scripts' / 'generate_figures.py')], check=True)
print('Figuras geradas em outputs/figuras/')